In [ ]:
# ==========================================
# 目的与作用：环境初始化、挂载云盘并加载全部JSON数据集，提取基础统计特征
# ==========================================
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import plotly.graph_objects as go
from torch.utils.data import Dataset, DataLoader

# 1. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/260509_dataset'

# 定义 11 种核心功能图层 (增加一个 empty 作为填充位)
ROOM_TYPES = [
    "empty", "entryway", "living_room", "dining_room", "kitchen",
    "bathroom", "bedroom", "corridor", "stairs", "balcony",
    "utility", "multi_purpose"
]
TYPE2ID = {t: i for i, t in enumerate(ROOM_TYPES)}
ID2TYPE = {i: t for i, t in enumerate(ROOM_TYPES)}

# 2. 加载数据
all_houses = []
for file in os.listdir(DATA_DIR):
    if file.endswith('.json'):
        with open(os.path.join(DATA_DIR, file), 'r', encoding='utf-8') as f:
            all_houses.append(json.load(f))

print(f"✅ 成功加载数据集，共 {len(all_houses)} 个户型。")

# 设定最大画布边界 (依据你的数据最大值并略作放宽)
MAX_X, MAX_Y, MAX_Z = 24000.0, 36000.0, 6000.0
MAX_ROOMS = 40 # 设定单个户型包含的最大房间数量上限（用于张量填充）

Mounted at /content/drive
✅ 成功加载数据集，共 96 个户型。


In [ ]:
# ==========================================
# 目的与作用：将JSON结构化数据转化为深度学习可计算的张量格式（归一化与独热编码），并构建PyTorch数据集
# ==========================================

class HouseDataset(Dataset):
    def __init__(self, houses):
        self.data = []
        self.conditions = []

        for house in houses:
            rooms = house.get('rooms', [])

            # 特征矩阵: [MAX_ROOMS, 12(类别) + 6(坐标)]
            house_tensor = np.zeros((MAX_ROOMS, len(ROOM_TYPES) + 6), dtype=np.float32)

            # 条件向量: 11个核心功能区的数量统计
            # 顺序: entryway, living, dining, kitchen, bath, bed, corridor, stairs, balcony, utility, multi
            condition = np.zeros(len(ROOM_TYPES) - 1, dtype=np.float32)

            for i, room in enumerate(rooms):
                if i >= MAX_ROOMS: break

                rtype = room['type']
                type_idx = TYPE2ID.get(rtype, 0)

                if type_idx > 0:
                    condition[type_idx - 1] += 1.0 # 记录房间数量作为条件

                # 归一化坐标 (0 到 1)
                xmin, ymin, zmin = room['box_min']
                xmax, ymax, zmax = room['box_max']

                coords = [
                    xmin / MAX_X, ymin / MAX_Y, zmin / MAX_Z,
                    xmax / MAX_X, ymax / MAX_Y, zmax / MAX_Z
                ]

                # 独热编码 (One-Hot)
                house_tensor[i, type_idx] = 1.0
                house_tensor[i, len(ROOM_TYPES):] = coords

            # 如果房间数不足 MAX_ROOMS，剩余部分默认 empty 类型 (索引0设为1)
            for j in range(len(rooms), MAX_ROOMS):
                house_tensor[j, 0] = 1.0

            self.data.append(house_tensor.flatten())
            self.conditions.append(condition)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx]), torch.tensor(self.conditions[idx])

dataset = HouseDataset(all_houses)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)
print(f"✅ 数据预处理完成！特征维度: {dataset[0][0].shape}, 条件维度: {dataset[0][1].shape}")

✅ 数据预处理完成！特征维度: torch.Size([720]), 条件维度: torch.Size([11])


In [ ]:
# ==========================================
# 目的与作用：构建条件变分自编码器（CVAE），实现“条件（如房间数量）注入”与“空间潜在特征”的联合解码
# ==========================================

class LayoutCVAE(nn.Module):
    def __init__(self, input_dim, cond_dim, latent_dim=128):
        super(LayoutCVAE, self).__init__()

        self.input_dim = input_dim
        self.cond_dim = cond_dim
        self.latent_dim = latent_dim

        # 编码器：输入(布局特征 + 约束条件)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim + cond_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

        # 解码器：输入(潜在向量 + 约束条件)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim + cond_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim),
            nn.Sigmoid() # 输出归一化到 0-1
        )

    def encode(self, x, c):
        inputs = torch.cat([x, c], dim=1)
        h = self.encoder(inputs)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, c):
        inputs = torch.cat([z, c], dim=1)
        return self.decoder(inputs)

    def forward(self, x, c):
        mu, logvar = self.encode(x, c)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z, c)
        return x_recon, mu, logvar

# 损失函数 (重构误差 MSE + 潜在分布差异 KLD)
def vae_loss(recon_x, x, mu, logvar):
    BCE = nn.functional.mse_loss(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + 0.1 * KLD # 0.1 是 Beta 权重，防止 KLD 在小数据集上崩盘

In [ ]:
# ==========================================
# 目的与作用：执行深度学习训练循环，让模型学习户型特征并拟合分布
# ==========================================

INPUT_DIM = MAX_ROOMS * (len(ROOM_TYPES) + 6) # 40 * (12+6) = 720
COND_DIM = len(ROOM_TYPES) - 1 # 11
LATENT_DIM = 64

model = LayoutCVAE(INPUT_DIM, COND_DIM, LATENT_DIM)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 300 # 因为数据量少，增加 Epoch 让其充分拟合

print("🚀 开始训练模型...")
model.train()
for epoch in range(EPOCHS):
    train_loss = 0
    for batch_x, batch_c in dataloader:
        optimizer.zero_grad()
        recon_batch, mu, logvar = model(batch_x, batch_c)
        loss = vae_loss(recon_batch, batch_x, mu, logvar)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()

    if (epoch+1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {train_loss / len(dataset):.4f}")

print("✅ 模型训练完成！")

🚀 开始训练模型...
Epoch [50/300], Loss: 16.1333
Epoch [100/300], Loss: 7.3607
Epoch [150/300], Loss: 4.2339
Epoch [200/300], Loss: 3.1872
Epoch [250/300], Loss: 2.6563
Epoch [300/300], Loss: 2.3925
✅ 模型训练完成！


In [ ]:
# ==========================================
# 目的与作用：根据用户需求（如3室2卫）执行推理生成，并强制应用300mm模数建筑规范
# ==========================================

def generate_layout(target_counts):
    model.eval()
    with torch.no_grad():
        # 1. 构造约束条件向量 (与前文顺序对应)
        # entryway, living, dining, kitchen, bath, bed, corridor, stairs, balcony, utility, multi
        c_val = [
            target_counts.get("entryway", 0), target_counts.get("living_room", 0),
            target_counts.get("dining_room", 0), target_counts.get("kitchen", 0),
            target_counts.get("bathroom", 0), target_counts.get("bedroom", 0),
            target_counts.get("corridor", 0), target_counts.get("stairs", 0),
            target_counts.get("balcony", 0), target_counts.get("utility", 0),
            target_counts.get("multi_purpose", 0)
        ]
        c_tensor = torch.tensor([c_val], dtype=torch.float32)

        # 2. 从潜空间随机采样“灵感”
        z = torch.randn(1, LATENT_DIM)

        # 3. 解码生成
        generated = model.decode(z, c_tensor).numpy().reshape(MAX_ROOMS, -1)

    # 4. 后处理：提取类别与坐标，应用 300mm 模数
    rooms_out = []
    for i in range(MAX_ROOMS):
        room_data = generated[i]
        type_probs = room_data[:12]
        type_idx = np.argmax(type_probs)

        if type_idx == 0: continue # 忽略 empty

        rtype = ID2TYPE[type_idx]
        coords = room_data[12:]

        # 还原物理尺寸
        xmin, ymin, zmin = coords[0]*MAX_X, coords[1]*MAX_Y, coords[2]*MAX_Z
        xmax, ymax, zmax = coords[3]*MAX_X, coords[4]*MAX_Y, coords[5]*MAX_Z

        # 强制模数对齐 (Round to nearest 300)
        def snap_300(val):
            return round(val / 300.0) * 300.0

        box_min = [snap_300(xmin), snap_300(ymin), snap_300(zmin)]
        box_max = [snap_300(xmax), snap_300(ymax), snap_300(zmax)]

        # 确保体积大于0
        if box_max[0] > box_min[0] and box_max[1] > box_min[1] and box_max[2] > box_min[2]:
            rooms_out.append({"type": rtype, "box_min": box_min, "box_max": box_max})

    return rooms_out

# 🎯 输入你的需求：例如生成 三室、一厅、一厨、两卫、一阳台
user_request = {
    "living_room": 1,
    "dining_room": 1,
    "bedroom": 3,
    "bathroom": 2,
    "kitchen": 1,
    "corridor": 2,
    "balcony": 1
}

new_layout = generate_layout(user_request)
print(f"✅ 成功生成布局，共产出 {len(new_layout)} 个有效功能体块。")

✅ 成功生成布局，共产出 22 个有效功能体块。


In [ ]:
# ==========================================
# 目的与作用：在 Notebook 中直接进行三维体素级别的可视化，通过色彩映射直观展示空间拓扑
# ==========================================

# 定义与 Rhino 类似的颜色映射规范
COLOR_MAP = {
    "entryway": "lightgrey",
    "living_room": "orange",
    "dining_room": "yellow",
    "kitchen": "red",
    "bedroom": "blue",
    "bathroom": "cyan",
    "corridor": "purple",
    "stairs": "magenta",
    "utility": "brown",
    "balcony": "lightblue",
    "multi_purpose": "pink"
}

def plot_3d_layout(rooms):
    fig = go.Figure()

    for r in rooms:
        x0, y0, z0 = r['box_min']
        x1, y1, z1 = r['box_max']
        color = COLOR_MAP.get(r['type'], "grey")

        # 绘制 Box 的 8 个顶点和 12 条边
        x_edges = [x0, x1, x1, x0, x0, x0, x1, x1, x0, x0, x1, x1, x1, x1, x0, x0]
        y_edges = [y0, y0, y1, y1, y0, y0, y0, y1, y1, y0, y0, y0, y1, y1, y1, y1]
        z_edges = [z0, z0, z0, z0, z0, z1, z1, z1, z1, z1, z1, z0, z0, z1, z1, z0]

        # 绘制网格框体
        fig.add_trace(go.Scatter3d(
            x=x_edges, y=y_edges, z=z_edges,
            mode='lines',
            line=dict(color=color, width=4),
            name=r['type']
        ))

        # 使用 Mesh3d 填充半透明颜色
        fig.add_trace(go.Mesh3d(
            x=[x0, x0, x1, x1, x0, x0, x1, x1],
            y=[y0, y1, y1, y0, y0, y1, y1, y0],
            z=[z0, z0, z0, z0, z1, z1, z1, z1],
            i=[7, 0, 0, 0, 4, 4, 6, 6, 4, 0, 3, 2],
            j=[3, 4, 1, 2, 5, 6, 5, 2, 0, 1, 6, 3],
            k=[0, 7, 2, 3, 6, 7, 1, 1, 5, 5, 7, 6],
            opacity=0.2,
            color=color,
            hoverinfo='text',
            text=f"类型: {r['type']}<br>尺寸: {x1-x0}x{y1-y0}x{z1-z0}"
        ))

    # 更新镜头与场景比例，锁定 Z 轴避免变形
    fig.update_layout(
        scene=dict(
            xaxis_title='X (mm)',
            yaxis_title='Y (mm)',
            zaxis_title='Z (mm)',
            aspectmode='data'
        ),
        title="AI 生成的 3D 功能体量分布图",
        width=900,
        height=700,
        showlegend=True
    )
    fig.show()

# 执行渲染
plot_3d_layout(new_layout)